# Data clean


In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv(r"D:\Jupiter\E-commerce_project\data\Ecommerce_messy.csv", encoding="UTF-8")


*duplicate values*

In [4]:
print(df.duplicated().sum())

df = df.drop_duplicates()

print(df.shape)
print(df.duplicated().sum())

20
(629, 19)
0


In [5]:
print(df.shape)
print(df.dtypes)
print(df.columns)


(629, 19)
order_id            object
customer_id         object
customer_name       object
age                 object
gender              object
city                object
state               object
product_id          object
product_name        object
category            object
price               object
quantity            object
discount_percent    object
sales               object
profit              object
payment_method      object
order_date          object
delivery_date       object
order_status        object
dtype: object
Index(['order_id', 'customer_id', 'customer_name', 'age', 'gender', 'city',
       'state', 'product_id', 'product_name', 'category', 'price', 'quantity',
       'discount_percent', 'sales', 'profit', 'payment_method', 'order_date',
       'delivery_date', 'order_status'],
      dtype='object')


In [6]:
df["customer_name"] = df["customer_name"].replace(
    ["-", "NA", "N/A", "?", "unknown", "None"],
    np.nan
)

missing_names = df["customer_name"].isna().sum()

df = df.dropna(subset=["customer_name"])
print(df["customer_name"].isna().sum)

<bound method Series.sum of 0      False
1      False
2      False
3      False
4      False
       ...  
644    False
645    False
646    False
647    False
648    False
Name: customer_name, Length: 604, dtype: bool>


*Age Column*

In [7]:
# print(df["age"].nunique())
# print(df["age"].isna().sum())
# print(df["age"].unique())
# print(df["age"].dtype)
df["age"] = df["age"].replace({
    "twenty five": 25,
    "25 years": 25,
    "?": np.nan,
    "unknown": np.nan
})

df["age"] = pd.to_numeric(df["age"], errors="coerce")

df.loc[(df["age"] < 15) | (df["age"] > 70), "age"] = np.nan

df["age"] = df["age"].fillna(df["age"].median()).astype(int)

# print(df["age"].unique())
# print(df["age"].isna().sum())


*Gender*

In [8]:
# print(df["gender"].nunique())
# print(df["gender"].unique())
# print(df["gender"].isnull().sum())
df["gender"] = df["gender"].str.replace(r"(?i)\s*male", "Male", regex=True)
df["gender"] = df["gender"].str.replace(r"(?i)\s*female", "Female", regex=True)
df["gender"] = df["gender"].replace({"M":"Male",
                                         "F":"Female"
                                         })
# print(df["gender"].unique())



*City*

In [9]:
# print(df["city"].nunique())
# print(df["city"].unique())
df["city"] = df["city"].str.lower().str.strip()
df["city"] = df["city"].str.title()
df["city"] = df["city"].replace({"Bombay":"Mumbai",
                                 "Bengaluru":"Bangalore",
                                 "-":"Unknown",
                                 "Delhi":"New Delhi"
                                 })
df["city"] = df["city"].fillna("Unknown")
# print(df["city"].isnull().sum())
# print((df["city"] == "Unknown").sum())
# print(df["city"].unique())


*state*

In [10]:
# print(df["state"].nunique())
# print(df["state"].unique())
# print(df["state"].isnull().sum())
df["state"] = df["state"].str.strip().str.title()




*

*product_id*

In [11]:
# print(df["product_id"].isnull().sum())
# print(df["product_id"].unique())


*product_name*

In [12]:
# print(df["product_name"].unique())
df["product_name"] = df["product_name"].str.title().str.strip()
# print(df["product_name"].isnull().sum())

*category*

In [13]:
# print(df["category"].nunique())
# print(df["category"].unique())
# print(df["category"].isnull().sum())
df["category"] = df["category"].str.strip().str.lower().str.title()

df["category"] = df["category"].replace({"Electronic":"Electronics",
                                             "Home And Kitchen":"Home & Kitchen",
                                             "Elec":"Electronics"})
# print(df["category"].unique())

*price*

In [14]:
# print(df["price"].isnull().sum())

df["price"] = df["price"].replace("free", 0)
df["price"] = df["price"].astype(str).str.replace(r"[₹,]", "", regex=True)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# print(df["price"].isnull().sum())
# print(df["price"].describe())

df.loc[df["price"] < 0, "price"] = np.nan
df.loc[df["price"] > 100000, "price"] = np.nan


print(df.loc[
    df["price"] == df["price"].max(),
    ["product_name", "category", "price"]
])

df["price"] = df["price"].fillna(df["price"].median())
# print(df["price"].isnull().sum())


    product_name     category     price
537       Tablet  Electronics  89885.05


*quantity*

In [15]:
# print(df["quantity"].nunique())
# print(df["quantity"].unique())
# print(df["quantity"].isnull().sum())
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df.loc[(df["quantity"] > 20) | (df["quantity"] <= 0)]
# print(df["quantity"].unique())

df["quantity"] = df["quantity"].fillna(df["quantity"].mode()[0])

# print(df["quantity"].isnull().sum())



*discount_percent*

In [16]:
# print(df["discount_percent"].nunique())
# print(df["discount_percent"].unique())
# print(df["discount_percent"].isnull().sum())

# print(df["discount_percent"].dtype)
df["discount_percent"] = df["discount_percent"].replace(r"(?i)\s*%", "", regex=True)
df["discount_percent"] = df["discount_percent"].replace(["?","-","abc","unknown", "-10"], np.nan)
df["discount_percent"] = df["discount_percent"].replace("10 percent",10)
df["discount_percent"] = pd.to_numeric(df["discount_percent"], errors="coerce")
df.loc[(df["discount_percent"] > 30), "discount_percent"] = np.nan
df["discount_percent"] = df["discount_percent"].fillna(df["discount_percent"].median()).astype(int)
# print(df["discount_percent"].nunique())
# print(df["discount_percent"].unique())
# print(df["discount_percent"].isnull().sum())

*sales*

In [17]:
# print(df["sales"].value_counts())
# print(df["sales"].unique())
print(df["sales"].nunique())
df["sales"] = df["sales"].replace(["error", "unknown"], np.nan)
df["sales"] = df["sales"].astype(str).str.replace(r"[₹,]", "", regex=True)
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")
df.loc[df["sales"] < 0, "sales"] = np.nan

# print(df[df["sales"] > 500000])
# df.loc[df["sales"] > 500000, ["price", "quantity", "discount_percent", "sales"]]

df["Expected_sales"] = df["price"] * df["quantity"] * (1 - df["discount_percent"] / 100)
df.loc[df["sales"] > 500000, ["price", "quantity", "discount_percent", "sales", "Expected_sales"]]

df["sales_difference"] =  (df["sales"] - df["Expected_sales"]).abs()
df.loc[df["sales"] > 500000, ["price", "quantity", "discount_percent", "sales", "Expected_sales", "sales_difference"]]
mask = (df["sales"] > 500000) & (df["sales_difference"] > 1)
df.loc[mask, "sales"] =  np.nan

df["sales"] = df["sales"].fillna(df["Expected_sales"])
# print(df["sales"].value_counts())
# print(df["sales"].unique())
print(df["sales"].isnull().sum())


573
0


*profit*

In [18]:
# print(df["profit"].isnull().sum())
# print(df["profit"].value_counts())
# print(df["profit"].unique())
# print(df["profit"].describe())
df["profit"] = df["profit"].replace(r"[₹,]", "", regex=True)
df["profit"] = pd.to_numeric(df["profit"], errors="coerce")
df.loc[(df["profit"] < -100000) | (df["profit"] > 100000),["sales", "profit"]]

mask = (df["profit"] < -100000) | (df["profit"] > 100000)
df.loc[mask, "profit"] = np.nan
df["profit"] = df["profit"].fillna(df["profit"].median())
print(df["profit"].isnull().sum())
print(df["profit"].describe())

0
count      604.000000
mean      4264.967334
std      12404.222554
min     -50000.000000
25%        151.295000
50%        898.630000
75%       3357.802500
max      94486.760000
Name: profit, dtype: float64


*payment_method*

In [19]:
# print(df["payment_method"].nunique())
# print(df["payment_method"].unique())
# print(df["payment_method"].isnull().sum())

df["payment_method"] = df["payment_method"].str.strip().str.lower().str.title()
df["payment_method"] = df["payment_method"].replace({"Upi":"UPI",
                                                     "Cash On Delivery":"COD",
                                                     "Cod":"COD"
                                                     })
df["payment_method"] = df["payment_method"].fillna("Unknown")
# print(df["payment_method"].unique())
# print(df["payment_method"].nunique())


*order data*

In [20]:
# print(df["order_date"].unique())
# print(df["order_date"].nunique())
# print(df["order_date"].isna().sum())

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", dayfirst=True)
df["order_date"] = df["order_date"].dt.strftime("%d-%m-%Y")
# print(df["order_date"].unique())
# print(df["order_date"].isna().sum())

C:\Users\rishi\AppData\Local\Temp\ipykernel_13376\1882375076.py:5: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", dayfirst=True)


*delivery_date*

In [21]:
# print(df["delivery_date"].unique())
# print(df["delivery_date"].isna().sum())
df["delivery_date"] = df["delivery_date"].replace("not available",np.nan)
df["delivery_date"] = pd.to_datetime(df["delivery_date"], errors="coerce", dayfirst=True)
df["delivery_date"] = df["delivery_date"].dt.strftime("%d-%m-%Y")
# print(df["delivery_date"].isna().sum())


*order status*

In [22]:
print(df["order_status"].nunique())
print(df["order_status"].unique())
df["order_status"] = df["order_status"].str.strip().str.lower().str.title()
df["order_status"] = df["order_status"].replace({"Delievered":"Delivered",
                                                     "Return":"Returned"})

print(df.loc[df["delivery_date"].isna(),"order_status"].value_counts(dropna=False))
print(df["order_status"].nunique())
print(df["order_status"].unique())

11
['Cancelled' 'Delivered' 'Pending' 'return' 'Delivered ' 'Returned'
 'Delievered' 'DELIVERED' 'delivered' ' Delivered ' 'CANCELLED']
order_status
Delivered    202
Cancelled     64
Returned      63
Pending       47
Name: count, dtype: int64
4
['Cancelled' 'Delivered' 'Pending' 'Returned']


*checklist*


In [ ]:
# ==================================================
# FINAL DATASET CHECKLIST
# ==================================================

# 1. Dataset Shape
print("SHAPE:")
print(df.shape)


# 2. Column Names
print("\nCOLUMN NAMES:")
print(df.columns.tolist())


# 3. Data Types
print("\nDATA TYPES:")
print(df.dtypes)


# 4. Dataset Information
print("\nDATASET INFO:")
df.info()


# 5. Missing Values
print("\nMISSING VALUES:")
print(df.isnull().sum())


# 6. Total Missing Values
print("\nTOTAL MISSING VALUES:")
print(df.isnull().sum().sum())


# 7. Duplicate Rows
print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())


# 8. Unique Values in Every Column
print("\nUNIQUE VALUES:")
print(df.nunique())


# 9. Numeric Columns Summary
print("\nNUMERIC COLUMNS SUMMARY:")
print(df.describe())


# 10. Categorical Columns Summary
print("\nCATEGORICAL COLUMNS SUMMARY:")
print(df.describe(include="object"))


# 11. Check Final Unique Values
categorical_columns = [
    "gender",
    "city",
    "state",
    "category",
    "payment_method",
    "order_status"
]

print("\nCATEGORICAL COLUMN VALUES:")

for col in categorical_columns:
    print(f"\n{col.upper()}:")
    print(df[col].unique())
    print("Missing:", df[col].isnull().sum())


# 12. Check Numeric Range
print("\nNUMERIC COLUMN MIN/MAX:")

numeric_columns = [
    "age",
    "price",
    "quantity",
    "discount_percent",
    "sales",
    "profit"
]

for col in numeric_columns:
    print(f"\n{col.upper()}")
    print("Min:", df[col].min())
    print("Max:", df[col].max())


# 13. Check Negative Values
print("\nNEGATIVE VALUE CHECK:")

for col in ["age", "price", "quantity", "discount_percent", "sales"]:
    print(f"{col} negative values:",
          (df[col] < 0).sum())


# Profit can be negative because of loss
print("profit negative values:",
      (df["profit"] < 0).sum())


# 14. Date Columns Check
print("\nDATE COLUMN CHECK:")

date_columns = [
    "order_date",
    "delivery_date"
]

for col in date_columns:
    print(f"\n{col}:")
    print("Missing:", df[col].isnull().sum())
    print("Data type:", df[col].dtype)


# 15. Final Random Sample
print("\nFINAL DATA SAMPLE:")
print(df.sample(10, random_state=42))

# Save the clean dataset

In [24]:
df.to_csv(r"D:\Jupiter\E-commerce_project\data\Ecommerce_clean.csv", index=False)